In [ ]:
import pickle

In [ ]:
with open("core_v2.pkl", "rb") as f:
    up_v2, dn_v2 = pickle.load(f)

with open("core_v3.pkl", "rb") as f:
    up_v3, dn_v3 = pickle.load(f)

In [ ]:
overlap_up = up_v2 & up_v3
overlap_dn = dn_v2 & dn_v3

print("UP overlap:", len(overlap_up))
print("DN overlap:", len(overlap_dn))

In [ ]:
print("UP retained:", len(overlap_up) / len(up_v3))
print("DN retained:", len(overlap_dn) / len(dn_v3))

In [ ]:
with open("core_v2_top100.pkl", "rb") as f:
    up_v2_top100, dn_v2_top100 = pickle.load(f)

with open("core_v2_top30.pkl", "rb") as f:
    up_v2_top30, dn_v2_top30 = pickle.load(f)

In [ ]:
overlap_up_top = up_v2_top100 & up_v2_top30
overlap_dn_top = dn_v2_top100 & dn_v2_top30

print("UP overlap:", len(overlap_up_top))
print("DN overlap:", len(overlap_dn_top))

In [ ]:
print("UP retained:", len(overlap_up_top) / len(up_v2_top30))
print("DN retained:", len(overlap_dn_top) / len(dn_v2_top30))

In [ ]:
import pandas as pd

df30 = pd.read_csv("core_scores_top30.csv")
df50 = pd.read_csv("core_scores_top50.csv")
df100 = pd.read_csv("core_scores_top100.csv")

In [ ]:
df = df50[["sig_id", "core_score"]].rename(columns={"core_score": "score50"})

df = df.merge(
    df30[["sig_id", "core_score"]].rename(columns={"core_score": "score30"}),
    on="sig_id"
)

df = df.merge(
    df100[["sig_id", "core_score"]].rename(columns={"core_score": "score100"}),
    on="sig_id"
)

In [ ]:
# =========================================================
# Robustness check for core_score across top30 / top50 / top100
# Using the same exported files as the directed-results notebooks
# =========================================================

from pathlib import Path
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# ---------------------------------------------------------
# 0. Paths
# ---------------------------------------------------------
PROJECT_ROOT = Path(".")
EXPORTS_DIR = PROJECT_ROOT / "data" / "exports"

EXP_PATH = r'..\data\exports\expression_matrix_clean.parquet'
META_PATH = r'..\data\exports\signature_metadata_clean.csv'

print("EXP_PATH:", EXP_PATH)
print("META_PATH:", META_PATH)

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
EXP = pd.read_parquet(EXP_PATH)
META = pd.read_csv(META_PATH)

print("EXP shape:", EXP.shape)
print("META shape:", META.shape)
print("META columns:", META.columns.tolist())

# ---------------------------------------------------------
# 2. Standardize expected columns
#    Adjust only if your column names differ
# ---------------------------------------------------------
# We expect:
# - one signature ID column in META
# - one cell line column in META
#
# The directed-results notebooks used standardization utilities,
# but here we do it manually to avoid extra dependencies.

possible_sig_cols = ["sig_id", "signature_id", "sigid"]
possible_cell_cols = ["cell_iname", "cell_id", "cell_line", "cell_mfc_name"]

sig_col = next((c for c in possible_sig_cols if c in META.columns), None)
cell_col = next((c for c in possible_cell_cols if c in META.columns), None)

if sig_col is None:
    raise ValueError(f"Could not find signature ID column in META. Available columns: {META.columns.tolist()}")

if cell_col is None:
    raise ValueError(f"Could not find cell line column in META. Available columns: {META.columns.tolist()}")

print("Using sig_col =", sig_col)
print("Using cell_col =", cell_col)

# ---------------------------------------------------------
# 3. Align EXP and META by signature
# ---------------------------------------------------------
# Assumption:
# - EXP rows are signatures
# - Either EXP.index contains sig_id or EXP has a sig_id column

if sig_col in EXP.columns:
    EXP = EXP.set_index(sig_col)
else:
    EXP.index.name = sig_col

META = META.drop_duplicates(subset=[sig_col]).set_index(sig_col)

common_ids = EXP.index.intersection(META.index)
EXP = EXP.loc[common_ids].copy()
META = META.loc[common_ids].copy()

print("Aligned EXP shape:", EXP.shape)
print("Aligned META shape:", META.shape)

# ---------------------------------------------------------
# 4. Keep only numeric gene columns
# ---------------------------------------------------------
gene_cols = EXP.select_dtypes(include="number").columns.tolist()
EXP_NUM = EXP[gene_cols].copy()

print("Numeric gene matrix shape:", EXP_NUM.shape)

# ---------------------------------------------------------
# 5. Compute effects_by_cell
#    mean moderated z-score per gene within each cell line
# ---------------------------------------------------------
effects_by_cell = EXP_NUM.groupby(META[cell_col]).mean().T
# rows = genes, cols = cell lines

print("effects_by_cell shape:", effects_by_cell.shape)
print("Cell lines:", effects_by_cell.columns.tolist())

# ---------------------------------------------------------
# 6. Helper to build consensus gene sets
# ---------------------------------------------------------
def build_consensus_gene_sets(effects_by_cell: pd.DataFrame, top_n: int = 50, min_votes: int = 2):
    """
    For each cell line:
      - select top_n most positive genes
      - select top_n most negative genes
    Then keep genes appearing in at least min_votes cell lines.

    Returns
    -------
    up_genes : list
    down_genes : list
    up_counts : pd.Series
    down_counts : pd.Series
    """
    up_votes = {}
    down_votes = {}

    for cell in effects_by_cell.columns:
        s = effects_by_cell[cell].dropna().sort_values(ascending=False)

        top_up = s.head(top_n).index.tolist()
        top_down = s.tail(top_n).index.tolist()

        for g in top_up:
            up_votes[g] = up_votes.get(g, 0) + 1

        for g in top_down:
            down_votes[g] = down_votes.get(g, 0) + 1

    up_counts = pd.Series(up_votes).sort_values(ascending=False)
    down_counts = pd.Series(down_votes).sort_values(ascending=False)

    up_genes = up_counts[up_counts >= min_votes].index.tolist()
    down_genes = down_counts[down_counts >= min_votes].index.tolist()

    return up_genes, down_genes, up_counts, down_counts

# ---------------------------------------------------------
# 7. Build top30 / top50 / top100 consensus sets
# ---------------------------------------------------------
up30, down30, up30_counts, down30_counts = build_consensus_gene_sets(
    effects_by_cell, top_n=30, min_votes=2
)

up50, down50, up50_counts, down50_counts = build_consensus_gene_sets(
    effects_by_cell, top_n=50, min_votes=2
)

up100, down100, up100_counts, down100_counts = build_consensus_gene_sets(
    effects_by_cell, top_n=100, min_votes=2
)

print("n_up30, n_down30  :", len(up30), len(down30))
print("n_up50, n_down50  :", len(up50), len(down50))
print("n_up100, n_down100:", len(up100), len(down100))

# ---------------------------------------------------------
# 8. Compute core_score
# ---------------------------------------------------------
def compute_core_score(expr_df: pd.DataFrame, up_genes, down_genes, score_name: str):
    up_present = [g for g in up_genes if g in expr_df.columns]
    down_present = [g for g in down_genes if g in expr_df.columns]

    if len(up_present) == 0:
        raise ValueError(f"No up genes found for {score_name}")
    if len(down_present) == 0:
        raise ValueError(f"No down genes found for {score_name}")

    score = expr_df[up_present].mean(axis=1) - expr_df[down_present].mean(axis=1)
    score.name = score_name
    return score

score30 = compute_core_score(EXP_NUM, up30, down30, "core_score_top30")
score50 = compute_core_score(EXP_NUM, up50, down50, "core_score_top50")
score100 = compute_core_score(EXP_NUM, up100, down100, "core_score_top100")

scores_df = pd.concat([score30, score50, score100], axis=1)
scores_df.index.name = sig_col
scores_df = scores_df.reset_index()

# add metadata back
scores_df = scores_df.merge(
    META.reset_index()[[sig_col, cell_col]],
    on=sig_col,
    how="left"
)

print(scores_df.head())
print(scores_df.shape)

# ---------------------------------------------------------
# 9. Global correlations
# ---------------------------------------------------------
comparisons = [
    ("core_score_top30", "core_score_top50"),
    ("core_score_top100", "core_score_top50"),
    ("core_score_top30", "core_score_top100"),
]

corr_rows = []

for x, y in comparisons:
    tmp = scores_df[[x, y]].dropna()

    pearson_r, pearson_p = pearsonr(tmp[x], tmp[y])
    spearman_rho, spearman_p = spearmanr(tmp[x], tmp[y])

    corr_rows.append({
        "comparison": f"{x} vs {y}",
        "n": len(tmp),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
    })

corr_df = pd.DataFrame(corr_rows)
print("\nGlobal correlations")
print(corr_df.round(4))

# ---------------------------------------------------------
# 10. Per-cell-line correlations
# ---------------------------------------------------------
per_cell_rows = []

for cell, subdf in scores_df.groupby(cell_col):
    for x, y in comparisons:
        tmp = subdf[[x, y]].dropna()
        if len(tmp) < 3:
            continue

        pearson_r, pearson_p = pearsonr(tmp[x], tmp[y])
        spearman_rho, spearman_p = spearmanr(tmp[x], tmp[y])

        per_cell_rows.append({
            "cell_line": cell,
            "comparison": f"{x} vs {y}",
            "n": len(tmp),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
        })

per_cell_corr_df = pd.DataFrame(per_cell_rows)
print("\nPer-cell-line correlations")
print(per_cell_corr_df.round(4))

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# Cargar scores ya calculados
df30 = pd.read_csv("core_scores_top30.csv")
df50 = pd.read_csv("core_scores_top50.csv")
df100 = pd.read_csv("core_scores_top100.csv")

# Unir por firma
df = (
    df30[["sig_id", "cell_id", "core_score"]]
    .rename(columns={"core_score": "score30"})
    .merge(
        df50[["sig_id", "core_score"]].rename(columns={"core_score": "score50"}),
        on="sig_id",
        how="inner"
    )
    .merge(
        df100[["sig_id", "core_score"]].rename(columns={"core_score": "score100"}),
        on="sig_id",
        how="inner"
    )
)

print(df.head())
print(df.shape)

# Correlaciones globales
comparisons = [
    ("score30", "score50"),
    ("score100", "score50"),
    ("score30", "score100"),
]

rows = []
for x, y in comparisons:
    pearson_r, pearson_p = pearsonr(df[x], df[y])
    spearman_rho, spearman_p = spearmanr(df[x], df[y])
    rows.append({
        "comparison": f"{x} vs {y}",
        "n": len(df),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
    })

corr_df = pd.DataFrame(rows)
print("\nGlobal correlations:")
print(corr_df.round(4))

# Correlaciones por línea celular
per_cell_rows = []
for cell, subdf in df.groupby("cell_id"):
    for x, y in comparisons:
        pearson_r, pearson_p = pearsonr(subdf[x], subdf[y])
        spearman_rho, spearman_p = spearmanr(subdf[x], subdf[y])
        per_cell_rows.append({
            "cell_id": cell,
            "comparison": f"{x} vs {y}",
            "n": len(subdf),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
        })

per_cell_corr_df = pd.DataFrame(per_cell_rows)
print("\nPer-cell-line correlations:")
print(per_cell_corr_df.round(4))